**Intelligent Vision Systems · Lab 6 · Camera Calibration**
Companion to **Lecture 6 — Camera Calibration**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_6/04-road-camera-birdseye.ipynb) &nbsp; [Lecture 6 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/06_Camera_Calibration.html#/demo-perspective) &nbsp; [All Lab 6 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-6-calibration)

# Lab 6D · Road Camera: Undistort, Bird's-Eye View & Curvature

_Lecture 6, Section 7: perspective transform and lane curvature._

Back to the self-driving car. We calibrate the **dash-cam** of the Udacity *Advanced Lane Lines* project, undistort road
frames, warp them to a **bird's-eye view** and measure the **lane curvature in metres**: the pipeline that follows on
from Lecture 5's straight lane lines.

In [ ]:
# Setup (only needed on Google Colab or a fresh environment).
# Uncomment the next line to install the packages this notebook uses:
# !pip install -q opencv-python numpy matplotlib

import os, urllib.request

# Fetch the lab files if they are not next to the notebook.
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_6"
for f in ['lab6_checks.py']:
    if not os.path.exists(f):
        os.makedirs(os.path.dirname(f) or ".", exist_ok=True)
        urllib.request.urlretrieve(f"{RAW}/{f}", f)

import lab6_checks as C
C.fetch_set('road_cal')
C.fetch_set('road')
C.fetch_set('gopro')
print("data ready")

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2
import lab6_checks as C
np.set_printoptions(precision=3, suppress=True)

files = sorted(glob.glob("data/camera_cal/*.jpg"))
for f in files[:5]:
    print(f, cv2.imread(f).shape)

## 1. Calibrate the road camera

This board has **9 × 6** inner corners. Careful: not every image has the same size, and some boards are cut off
by the frame. Make sure every view you use is **1280 × 720**.

In [ ]:
nx, ny = 9, 6
objp = C.board_points(nx, ny)
objpoints, imgpoints = [], []
for f in files:
    gray = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    # TODO: handle images that are not 1280 x 720
    ok, corners = C.find_corners(gray, (nx, ny))
    if ok:
        objpoints.append(objp); imgpoints.append(corners)

ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, (1280, 720), None, None)
print(len(objpoints), "views, RMS", ret); print(mtx); print(dist)

In [ ]:
C.check_road_calibration(mtx, dist)

In [ ]:
road = cv2.imread("data/road/test5.jpg")
und = cv2.undistort(road, mtx, dist, None, mtx)
fig, ax = plt.subplots(1, 3, figsize=(24, 5))
ax[0].imshow(road[:, :, ::-1]); ax[0].set_title("raw")
ax[1].imshow(und[:, :, ::-1]); ax[1].set_title("undistorted")
ax[2].imshow(cv2.absdiff(road, und).max(2) * 3, cmap="magma"); ax[2].set_title("|difference| x 3")
for a in ax: a.axis("off")
plt.show()

## 2. Quiz: undistort and transform a chessboard (Udacity *corners_unwarp*)

Write `corners_unwarp(img, nx, ny, mtx, dist)` that:
1. undistorts with `cv2.undistort`, converts to grayscale and finds the 8 × 6 corners (GoPro image);
2. uses the **outer four** detected corners as `src`;
3. maps them to a rectangle `dst` **in the same order**, offset 100 px from the image border;
4. returns `warped, M` (`cv2.getPerspectiveTransform` + `cv2.warpPerspective`).

Tip: `corners.reshape(-1, 2)` makes indexing like `corners[nx - 1]` work in OpenCV 4 **and** 5.

In [ ]:
K_gopro, D_gopro = C.reference_gopro()       # the GoPro calibration from Lab 6C

def corners_unwarp(img, nx, ny, mtx, dist):
    # TODO
    M = None
    warped = np.copy(img)
    return warped, M

img = cv2.imread("data/calibration_wide/GOPR0062.jpg")
top_down, perspective_M = corners_unwarp(img, 8, 6, K_gopro, D_gopro)
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 7.5))
ax1.imshow(img[:, :, ::-1]); ax1.set_title("Original Image", fontsize=22)
ax2.imshow(top_down[:, :, ::-1]); ax2.set_title("Undistorted and Warped Image", fontsize=22)
plt.show()

In [ ]:
C.check_corners_unwarp(corners_unwarp)

<details><summary><b>Show the Udacity solution</b> (try first!)</summary>

```python
def corners_unwarp(img, nx, ny, mtx, dist):
    undist = cv2.undistort(img, mtx, dist, None, mtx)
    gray = cv2.cvtColor(undist, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, (nx, ny), None)
    warped, M = undist, None
    if ret:
        corners = corners.reshape(-1, 2)
        cv2.drawChessboardCorners(undist, (nx, ny), corners, ret)
        offset = 100
        img_size = (gray.shape[1], gray.shape[0])
        src = np.float32([corners[0], corners[nx - 1], corners[-1], corners[-nx]])
        dst = np.float32([[offset, offset], [img_size[0] - offset, offset],
                          [img_size[0] - offset, img_size[1] - offset], [offset, img_size[1] - offset]])
        M = cv2.getPerspectiveTransform(src, dst)
        warped = cv2.warpPerspective(undist, M, img_size)
    return warped, M
```
</details>

## 3. Bird's-eye view of the road

On `straight_lines1.jpg` (a straight road), choose four `src` points on the two lane lines (a trapezoid) and map them to
a rectangle. Good points make both lane lines **vertical and parallel** in the warped image.
Hover over the image (or use `%matplotlib widget`) to read coordinates.

In [ ]:
straight = cv2.undistort(cv2.imread("data/road/straight_lines1.jpg"), mtx, dist, None, mtx)
H, W = straight.shape[:2]

# TODO: pick better source points (top-left, top-right, bottom-right, bottom-left)
src = np.float32([[500, 480], [780, 480], [1100, 720], [200, 720]])
dst = np.float32([[300, 0], [980, 0], [980, 720], [300, 720]])

M = cv2.getPerspectiveTransform(src, dst)
Minv = cv2.getPerspectiveTransform(dst, src)
bev = cv2.warpPerspective(straight, M, (W, H), flags=cv2.INTER_LINEAR)

a = straight.copy(); cv2.polylines(a, [src.astype(np.int32)], True, (0, 0, 255), 3)
b = bev.copy(); cv2.polylines(b, [dst.astype(np.int32)], True, (0, 0, 255), 3)
fig, ax = plt.subplots(1, 2, figsize=(22, 6.5))
ax[0].imshow(a[:, :, ::-1]); ax[1].imshow(b[:, :, ::-1])
for x in ax: x.axis("off")
plt.show()

In [ ]:
C.check_birdseye(M)

## 4. Lane pixels and curvature on a curved road

For a near-vertical lane line we fit $x = f(y) = Ay^2 + By + C$ and compute the radius of curvature at the bottom of the
image (closest to the car):

$$R = \frac{(1 + (2Ay + B)^2)^{3/2}}{|2A|}$$

To get **metres**, convert pixel coordinates **before** fitting. For this camera and warp, about 30 m of road fills the
720 rows and the 3.7 m lane is ~700 px wide (the `dst` rectangle above is 680 px).

In [ ]:
curve = cv2.undistort(cv2.imread("data/road/test2.jpg"), mtx, dist, None, mtx)
bev = cv2.warpPerspective(curve, M, (W, H))
hls = cv2.cvtColor(bev, cv2.COLOR_BGR2HLS)
mask = ((hls[:, :, 2] > 120) & (hls[:, :, 1] > 90)) | (hls[:, :, 1] > 205)   # yellow or white paint (Lecture 5!)
ys, xs = np.nonzero(mask)
left, right = xs < W // 2, xs >= W // 2
ym_per_px, xm_per_px = 30 / 720, 3.7 / 700

def radius_m(ys, xs, y_eval=H - 1):
    # TODO: fit a 2nd-order polynomial x(y) in METRES and return the radius at y_eval
    return 0.0

R_left, R_right = radius_m(ys[left], xs[left]), radius_m(ys[right], xs[right])
print(f"left: {R_left:.0f} m   right: {R_right:.0f} m")

plt.figure(figsize=(14, 7)); plt.imshow(mask, cmap="gray")
yy = np.arange(H)
for sel, col in ((left, "red"), (right, "cyan")):
    A, B, C0 = np.polyfit(ys[sel], xs[sel], 2); plt.plot(A * yy**2 + B * yy + C0, yy, color=col, lw=3)
plt.title("lane pixels in the bird's-eye view + polynomial fits"); plt.axis("off"); plt.show()

In [ ]:
C.check_curvature(R_left, R_right)

## 5. Back to the road: draw the lane on the original frame

`Minv` takes the bird's-eye drawing back to the camera view.

In [ ]:
lane = np.zeros_like(bev)
fits = [np.polyfit(ys[s], xs[s], 2) for s in (left, right)]
lx = np.polyval(fits[0], yy); rx = np.polyval(fits[1], yy)
pts = np.vstack([np.stack([lx, yy], 1), np.stack([rx, yy], 1)[::-1]]).astype(np.int32)
cv2.fillPoly(lane, [pts], (0, 200, 0))
back = cv2.warpPerspective(lane, Minv, (W, H))
out = cv2.addWeighted(curve, 1, back, 0.35, 0)
plt.figure(figsize=(14, 8)); plt.imshow(out[:, :, ::-1]); plt.axis("off"); plt.show()

## Your turn

1. Warp `test2.jpg` **without** undistorting first (use the raw frame). How much does the curvature change? Which number do
   you trust, and why?
2. Compute the car's **offset from the lane centre** in metres (assume the camera is mounted at the image centre).
3. Run sections 4–5 on `test3.jpg`, `test5.jpg` and `test6.jpg`. Where does the simple colour mask fail? (This becomes the
   Advanced Lane Finding pipeline.)